# Fig. 2b — Animal data: reward magnitude & learning efficiency

Reproductions of two panels from **Gong, Martell, Dudman & Coddington,
*Reward magnitude determines reinforcement learning efficiency*, Science 392, eaeb0813 (2026)**,
for side-by-side comparison with the SSB model.

**Gong et al. Fig. 1E** (our Fig. 2b left) — hidden-target navigation task: *reinforcements to criterion* (a sample-efficiency
measure) as a function of reward size (5–100 µL). Larger rewards → far fewer reinforcements to
reach trained performance.

**Gong et al. Fig. 4B** (our Fig. 2b right) — effort-based reach-to-pull joystick task: percentage of successfully executed
trials as a function of training session, interleaved-large vs standard reward.

### Data provenance & caveats
- **Gong et al. Fig. 1E** uses per-mouse target-latency sequences extracted from the public navigation
  dataset (figshare `10.25378/janelia.32145868`, *figshareSV.mat*), embedded below as base64 so
  the notebook is self-contained. The authors' *exact* "90% of final trained performance"
  algorithm is not in the public code, so the criterion here is a **documented reconstruction**
  (90% of the drop from initial → asymptotic latency, on a causally-smoothed curve). It
  reproduces the published monotonic trend and approximate magnitudes; the smoothing window and
  criterion fraction are exposed as parameters for tuning.
- **Gong et al. Fig. 4B** (our Fig. 2b right) was traced by hand from the published panel in Illustrator; there is no code for it.

Figure style follows the project's Nature conventions (small panels, 7–9 pt fonts, thin axes,
few ticks). Panels save as SVG next to this notebook.

In [ ]:
import io, base64
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap

mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['svg.fonttype'] = 'none'   # keep text as text in the SVG

# ---- plotting helpers ----
def nature_ax(ax, xlabel='', ylabel='', nticks=4, nx=None, ny=None):
    for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
    for sp in ('left', 'bottom'): ax.spines[sp].set_linewidth(0.25)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nx or nticks))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(ny or nticks))
    ax.tick_params(labelsize=7, width=0.25)
    if xlabel: ax.set_xlabel(xlabel, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, fontsize=9)

# ---- style_ax: matches the model panel f2_sample_efficiency.svg ----
def style_ax(ax, fontsize=8):
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    for sp in ax.spines.values(): sp.set_linewidth(0.5)
    ax.tick_params(axis='both', labelsize=fontsize, length=2, pad=1.5, width=0.5)

## Gong et al. Fig. 1E (our Fig. 2b left) — navigation sample efficiency

Each `svMice{R}_{i}` array holds the successive **target latencies** (s) on successful
trajectories for mouse `i` in reward group `R` µL, concatenated across all training.
Reward groups: 5, 10, 20, 50, 100 µL (n = 14, 5, 5, 5, 14 mice).

In [ ]:
# per-mouse navigation latencies extracted from figshare 10.25378/janelia.32145868 (figshareSV.mat)
d = np.load('../../data/animals_Gong/gong2026_nav_latencies.npz', allow_pickle=True)
REWARDS = [5, 10, 20, 50, 100]

def smooth_causal(x, w):
    x = np.asarray(x, float); n = len(x); out = np.full(n, np.nan)
    for i in range(n):
        s = x[max(0, i-w+1):i+1]; s = s[np.isfinite(s)]
        if len(s): out[i] = np.mean(s)
    return out

# Reinforcements until the smoothed target latency has fallen `crit_frac` of the way
# from the initial to the asymptotic (trained) level. Reconstruction of the Fig 1E metric
# ('reinforcements before achieving 90% of the final trained performance'); tune
# `window`/`crit_frac` -- the monotonic trend across reward sizes is robust to these choices.
def reinforcements_to_criterion(lat, window=50, crit_frac=0.90,
                                head_frac=0.15, tail_frac=0.10):
    lat = np.asarray(lat, float); n = len(lat)
    sm = smooth_causal(lat, window)
    A  = np.nanmedian(sm[-max(3, int(n*tail_frac)):])   # asymptotic (trained) latency
    B0 = np.nanmedian(sm[:max(3, int(n*head_frac))])    # initial latency
    if not np.isfinite(B0) or B0 <= A:
        return 1        # already at trained performance -> criterion met by the 1st reinforcement
    thr = A + (1 - crit_frac) * (B0 - A)
    idx = np.where(sm <= thr)[0]
    return int(idx[0]) + 1 if len(idx) else n   # counts are 1-based: >=1 reinforcement

per_group = {}
for R in REWARDS:
    n = int(d[f'n_svMice{R}'])
    per_group[R] = np.array([reinforcements_to_criterion(d[f'svMice{R}_{i}'])
                             for i in range(n)], float)
    v = per_group[R]
    print(f"R={R:3d} µL  n={n:2d}   mean={np.nanmean(v):4.0f}  "
          f"SEM={np.nanstd(v)/np.sqrt(n):3.0f}  median={np.nanmedian(v):4.0f}")

In [ ]:
# Styled to match the model panel f2_sample_efficiency.svg: black open circles
# jittered at each reward value, red '_' mean +/- SEM, thin spines (style_ax).
fig, ax = plt.subplots(figsize=(2.0, 1.4))
rng = np.random.default_rng(0)
for R in REWARDS:
    v = per_group[R]; v = v[np.isfinite(v)]
    jitter = rng.normal(0, 0.015*R, size=len(v))            # jitter proportional to reward
    ax.scatter(np.full(len(v), R) + jitter, v,
               facecolors='none', edgecolors='k', s=8, alpha=0.7, lw=0.5, zorder=2)
    m, sem = np.mean(v), np.std(v)/np.sqrt(len(v))
    ax.errorbar(R, m, yerr=sem, fmt='_', color='red', markersize=6,
                capsize=2, markeredgewidth=1, elinewidth=1, zorder=3)

ax.set_xlabel('Reward size (µL)', fontsize=8)
ax.set_ylabel('Reinforcements\nto criterion', fontsize=8)
style_ax(ax)
ax.set_ylim(0, None)
fig.tight_layout(pad=0.3)
plt.savefig('f2_animals_fig1E_sample_efficiency.svg', dpi=300, bbox_inches='tight')
plt.show()